# Uplift modeling in marketing for treatment targeting

Application to the Hillstrom MineThatData E-Mail Analytics Challenge (2008)

Data scientist: Enio Rubens

Version: v1.0

Dataset: Kevin Hillstrom, MineThatData E-Mail Analytics Challenge (2008)

Stack: `econml`, `causalml`, `sklift`, `mlflow`

This notebook builds a causal targeting analysis for e-mail marketing, empirically showing why ranking customers by propensity (conversion probability) wastes budget and how uplift modeling identifies the customers for whom treatment changes the outcome.

## Notebook 1. Framing and EDA

Note on structure: this project was split into multiple notebooks, one per section group, to keep each file within an editable size (the S1-S4 history originally lived in a single notebook, now archived under `notebooks/_superseded/`). Each notebook reloads deterministically what it needs from earlier sections, with no shared kernel state across files. A cross-notebook index will be added once the split is complete; for now, the list below serves as a reading reference for the remaining notebooks.

## Table of contents (internal links only in sections 1 and 2)

1. [Framing and setup](#s1)
    1.1 [Technical setup](#s1-1)
    1.2 [Context and motivation](#s1-2)
    1.3 [The 4 quadrants of treatment response](#s1-3)
    1.4 [Causal formalization](#s1-4)
    1.5 [Dataset: Hillstrom MineThatData](#s1-5)
    1.6 [Roadmap](#s1-6)
2. [EDA and randomization audit](#s2)
    2.1 [Treatment distribution](#s2-1)
    2.2 [Univariate distributions](#s2-2)
    2.3 [Randomization audit](#s2-3)
    2.4 [Outcomes: distribution and raw rates](#s2-4)
    2.5 [ATE with confidence intervals](#s2-5)
    2.6 [Internal-validity synthesis](#s2-6)
3. [Response-targeting baseline](02_Baseline_Propensity_EN.ipynb)
4. [Meta-learners (S/T/X/R)](03_Meta_Learners_EN.ipynb)
5. [Direct uplift models (Causal Forest, Uplift Trees)](04_Causal_Forest_Uplift_Trees_EN.ipynb)
6. [Evaluation (Qini, AUUC, uplift@k)](05_Evaluation_Sealed_Test_EN.ipynb)
7. [Heterogeneity and uplift funnel](06_Heterogeneity_Uplift_Funnel_EN.ipynb)
8. [Policy learning and ROI simulation](07_Policy_Learning_ROI_EN.ipynb)
9. [Robustness and limitations](08_Robustness_Limitations_EN.ipynb)

<a id='s1'></a>
# Section 1. Framing and setup

This section positions the business problem of budget-constrained targeting, establishes the conceptual and financial distinction between propensity and uplift, introduces the 4-quadrant treatment-response framework, formalizes the required causal concepts (ITE, ATE, and CATE), loads the Hillstrom dataset with justification for the primary outcome, and maps the remaining sections.

<a id='s1-1'></a>
## 1.1 Technical setup

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.exceptions import ConvergenceWarning

# Path bootstrap: allows `from src...` from the notebooks directory.
PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import SEED
from src.i18n import make_lang
from src.viz import apply_plot_style

np.random.seed(SEED)
pd.set_option('display.max_columns', 50)
pd.set_option('display.precision', 4)
warnings.filterwarnings('ignore', category=FutureWarning)
# The internal causalml meta-learner propensity model (S4) uses an
# elastic-net solver that does not converge within the default max_iter on
# small samples. This does not affect the result because it is only a
# nuisance model, but it pollutes the output.
warnings.filterwarnings('ignore', category=ConvergenceWarning)

# Idioma canônico deste notebook é PT — passthrough, sem chamada de rede.
# EN edition: only this line is switched to make_lang('en').
lang = make_lang('en')

apply_plot_style()

In [ ]:
# Stack causal (verifique a instalação antes da primeira run)
# pip install econml causalml scikit-uplift shap mlflow

# Meta-learners e árvores de uplift
from causalml.inference.meta import BaseSRegressor, BaseTRegressor, BaseXRegressor, BaseRRegressor
from causalml.inference.tree import UpliftRandomForestClassifier

# Causal Forest com intervalos de confiança
from econml.dml import CausalForestDML
from econml.metalearners import TLearner, SLearner, XLearner

# Avaliação
from sklift.metrics import uplift_at_k, qini_auc_score, uplift_auc_score

from src.compat import patch_sklearn_matplotlib_support
patch_sklearn_matplotlib_support()
from sklift.viz import plot_qini_curve, plot_uplift_curve

# Interpretabilidade
import shap

In [ ]:
from src.config import ARTIFACTS_DIR, RETRAIN

ARTIFACTS_DIR.mkdir(exist_ok=True, parents=True)

# MLflow — ajuste o URI conforme seu setup Windows
# mlflow.set_tracking_uri('file:///C:/path/to/mlruns')
# mlflow.set_experiment('uplift_hillstrom')

<a id='s1-2'></a>
## 1.2 Context and motivation

### The business problem

A retail company operates an e-mail marketing channel with positive cost (deliverability, list fatigue, and opt-outs) and a finite budget. The recurring data-science question asks which customers to include in the next campaign. The standard propensity-modeling approach estimates $P(\text{conversion} \mid X)$ and contacts the top-k%, ranking customers by conversion likelihood without separating those who respond only because of the e-mail.

### The causal question

What matters for campaign return on investment is identifying who converts because of the e-mail, that is, the incremental treatment effect per customer. This distinction separates predictive analytics (what will happen) from prescriptive and causal analytics (what action to take), where uplift modeling belongs (Gutierrez & Gérardy, 2017).

### Connection with previous portfolio projects

| Project | Question answered | Category |
|---|---|---|
| Customer Churn Prediction | Who will cancel? | Predictive (propensity) |
| Customer Segmentation + NBA | To whom should each action be offered? | Prescriptive (without causal identification) |
| Marketing Campaign Optimization | Which campaign has the best average performance? | Predictive and descriptive |
| Uplift Modeling (this project) | For whom does treatment make a difference? | Causal |

This notebook covers the causal estimation step for heterogeneous treatment effects.

<a id='s1-3'></a>
## 1.3 The 4 quadrants of treatment response

For any customer $i$ and binary treatment, two potential outcomes are defined (Rubin, 1974): $Y_i(1)$, the outcome if the customer is treated, and $Y_i(0)$, the outcome if the customer is not treated. Crossing these binary outcomes defines four customer types (Devriendt et al., 2018):

|  | $Y(0) = 0$ (does not convert without treatment) | $Y(0) = 1$ (converts without treatment) |
|---|---|---|
| $Y(1) = 1$ (converts if treated) | Persuadables (positive uplift; drive campaign return) | Sure Things (convert regardless; treating them wastes budget) |
| $Y(1) = 0$ (does not convert if treated) | Lost Causes (never convert; treating them is wasteful) | Sleeping Dogs / Do Not Disturb (treatment reduces conversion; propensity modeling fails to detect them) |

### Practical implications

Propensity modeling ranks customers by $P(Y=1 \mid X)$, placing Sure Things at the top and spending budget on customers who would convert anyway. Sleeping Dogs occur in real domains when customers become irritated by the campaign, churn, or turn to alternatives; propensity modeling treats them as good candidates, whereas uplift identifies them as negative cost. Only Persuadables generate positive marginal campaign return, making them the target of uplift modeling.

### Why average ATE hides heterogeneity

An average conversion ATE of +2pp can arise from two distinct scenarios: in the homogeneous scenario (Scenario A), every customer experiences a +2pp effect and broad targeting is reasonable; in the heterogeneous scenario (Scenario B), 10% of customers have a +20pp effect and 90% have 0pp, so targeting only the responsive 10% quadruples campaign efficiency. Although both yield the same average ATE, only the heterogeneous scenario supports budget optimization by estimating the full distribution of individual effects.

<a id='s1-4'></a>
## 1.4 Causal formalization

### Notation

In the problem notation, $T_i \in \{0, 1\}$ denotes the treatment indicator (no e-mail or e-mail sent), $Y_i$ represents the observed outcome (`visit`, `conversion`, or `spend`), $X_i$ denotes the vector of pre-treatment covariates, and $Y_i(t)$ denotes the potential outcome under treatment $t$.

### Effects

The Individual Treatment Effect (ITE) measures the individual gain:

$$\tau_i = Y_i(1) - Y_i(0)$$

By the Fundamental Problem of Causal Inference (Holland, 1986), $\tau_i$ is unobservable for any individual because $Y_i(0)$ and $Y_i(1)$ are never observed simultaneously.

The Average Treatment Effect (ATE) measures the population mean gain:

$$\tau = \mathbb{E}[Y(1) - Y(0)]$$

Under randomization (Imbens & Rubin, 2015), ATE is estimable as the difference in means between arms and matches the quantity reported in standard A/B tests.

The Conditional Average Treatment Effect (CATE) is the target estimated by uplift modeling:

$$\tau(x) = \mathbb{E}[Y(1) - Y(0) \mid X = x]$$

### Causal identification

Under randomization (verified in Section 2 via balance checks), unconfoundedness is guaranteed by design:

$$\{Y(0), Y(1)\} \perp T \mid X$$

Therefore, CATE is identifiable from observable quantities:

$$\tau(x) = \mathbb{E}[Y \mid T=1, X=x] - \mathbb{E}[Y \mid T=0, X=x]$$

The methods in Sections 4 and 5 (S-, T-, X-, R-learner, and Causal Forest) implement distinct strategies to estimate these conditional expectations, each carrying different bias-variance trade-offs.

<a id='s1-5'></a>
## 1.5 Dataset: Hillstrom MineThatData (2008)

The dataset published by Kevin Hillstrom (2008) in the [MineThatData](http://blog.minethatdata.com/2008/03/minethatdata-e-mail-analytics-and-data.html) challenge is a canonical benchmark for uplift modeling in marketing, cited across the literature (Radcliffe & Surry 2011; Diemert et al. 2018; Olaya et al. 2020). The experiment includes 64,000 customers who purchased in the prior 12 months, randomized across three arms (No E-Mail as control, Mens E-Mail Campaign, and Womens E-Mail Campaign), with a 2-week post-send observation window.

In [ ]:
from src.data import load_hillstrom

df = load_hillstrom()
print(f'Shape: {df.shape}')
print(f'\nColunas:')
for c, dt in zip(df.columns, df.dtypes):
    print(f'  {c:20s} {str(dt)}')
df.head()

### Dictionary of variables

Pre-treatment covariates ($X$):

| Variable | Type | Description |
|---|---|---|
| `recency` | int | Months since last purchase (1–12) |
| `history` | float | Amount spent (USD) in the last 12 months |
| `history_segment` | cat | Historical spending range (7 levels) |
| `mens` | bin | Purchased male product in the last year |
| `womens` | bin | Purchased female product in the last year |
| `zip_code` | cat | Rural / Suburban / Urban |
| `newbie` | bin | New customer (≤12 months) |
| `channel` | cat | Phone / Web / Multichannel |

Treatment ($T$):

| Variable | Values |
|---|---|
| `segment` | No E-Mail / Mens E-Mail / Womens E-Mail |

Outcomes ($Y$):

| Variable | Type | Observed rate |
|---|---|---|
| `visit` | bin | 14.7% visited the site |
| `conversion` | bin | 0.9% purchased |
| `spend` | float | zero mass, continuous among converters |

In [ ]:
from src.config import ARMS, TREATMENT_COL
from src.eda import outcome_summary_by_arm

summary = outcome_summary_by_arm(df, TREATMENT_COL, ARMS)
labels = lang({'header': 'ATE naive (Mens vs No E-Mail):'})
print(summary[['n', 'visit_rate', 'conversion_rate', 'spend_mean']])
print(f"\n{labels['header']}")
print(f'  visit:      {summary.loc["Mens E-Mail", "visit_rate"] - summary.loc["No E-Mail", "visit_rate"]:+.4f}')
print(f'  conversion: {summary.loc["Mens E-Mail", "conversion_rate"] - summary.loc["No E-Mail", "conversion_rate"]:+.4f}')
print(f'  spend:      ${summary.loc["Mens E-Mail", "spend_mean"] - summary.loc["No E-Mail", "spend_mean"]:+.4f}')

### Primary outcome selection

The dataset provides three outcomes ordered by proximity to the purchase funnel: `visit` (14.7% positives, top-of-funnel engagement), `conversion` (0.9% positives, mid/bottom-of-funnel), and `spend` (mass at zero, continuous among converters, monetary value).

In this analysis, `visit` serves as the primary outcome for three reasons. Statistically, the 0.9% positive rate in `conversion` makes subgroup CATE estimation noisy, because rare outcomes require hundreds of thousands of observations per arm, exceeding the available sample size. In the literature, Radcliffe & Surry (2011), Diemert et al. (2018), and official CausalML tutorials likewise use `visit` as the primary outcome for the same reason. From a business standpoint, uplift in visits provides a proxy for incremental engagement generated by the campaign.

`conversion` and `spend` are treated as secondary outcomes in the dedicated funnel analysis in Section 7, where divergence between outcome-specific uplift rankings shows that top-of-funnel uplift differs from bottom-of-funnel uplift and that optimizing for visits can diverge from optimizing for revenue.

<a id='s1-6'></a>
## 1.6 Notebook roadmap

| Section | Content | Output |
|---|---|---|
| S1 | Causal framing and setup | This section |
| S2 | EDA and randomization audit | Experimental design confirmation |
| S3 | Baseline: response-targeting baseline | Benchmark for comparison |
| S4 | Meta-learners (S/T/X/R) | 4 CATE estimators |
| S5 | Direct uplift models (Causal Forest, Uplift Trees) | Direct estimators and confidence intervals |
| S6 | Evaluation (Qini, AUUC, uplift@k) | Model comparison |
| S7 | Heterogeneity analysis, quantile profiles, and uplift funnel | Profile identification of Persuadables |
| S8 | Policy learning and ROI simulation | Targeting policy and economic scenarios |
| S9 | Robustness and limitations | Sensitivity analysis and caveats |

The expected final output consists of actionable recommendations on which customer profiles should receive treatment in a future campaign, supported by confidence intervals and ROI simulation.

<a id='s2'></a>
# Section 2. EDA and randomization audit

This section characterizes covariate and outcome distributions in the study population, audits randomization before making any causal claims, estimates naive ATEs with confidence intervals as a benchmark for subsequent CATE estimators, and diagnoses the low frequency of `conversion` that motivated selecting `visit` as the primary outcome.

In causal inference, identification precedes any causal claim. In a randomized controlled trial, identification comes from the design because randomization guarantees $\{Y(0), Y(1)\} \perp T \mid X$. Because implementation bugs, differential attrition, or imperfect stratification can compromise covariate balance in collected data, formal statistical tests verify that the Hillstrom experiment preserves the expected balance.

In [ ]:
# Vetores de variáveis para uso recorrente nas próximas seções (fonte única: src/config.py)
from src.config import (
    ARMS, BIN_VARS, CAT_VARS, CONT_VARS, CONTROL_ARM, OUTCOMES,
    TREATED_ARMS, TREATMENT_COL as TREATMENT,
)

<a id='s2-1'></a>
## 2.1 Treatment distribution

The first check verifies whether the three arms have comparable sizes. In a 1:1:1 RCT, approximately 33% of observations are expected in each arm; small fluctuations stem from simple random assignment, whereas large deviations would signal a design issue.

In [ ]:
from src.eda import treatment_distribution

treat_table, chi2, p = treatment_distribution(df, TREATMENT, ARMS)
labels = lang({'header': 'Chi-squared vs uniforme 1:1:1'})
print(treat_table)
print(f"\n{labels['header']}: chi2={chi2:.4f}, p={p:.4f}")

The arm distribution is nearly uniform (≈33.3% per arm), and the high p-value confirms that observed deviations are consistent with random allocation without evidence of assignment errors.

<a id='s2-2'></a>
## 2.2 Univariate distributions

Before the balance check, univariate distributions summarize the study population.

In [ ]:
# Continuous variables
from src.viz import plot_univariate_continuous

labels = lang({
    'header': 'Variáveis contínuas',
    'dist_prefix': 'Distribuição de',
    'title': 'Distribuições univariadas — variáveis contínuas',
    'subtitle': 'history tem cauda longa; recency é aproximadamente uniforme entre 1 e 12 meses',
})
print(f"=== {labels['header']} ===")
print(df[CONT_VARS].describe().round(2))

fig, axes = plot_univariate_continuous(
    df, CONT_VARS, titles=[f"{labels['dist_prefix']} {v}" for v in CONT_VARS],
    title=labels['title'], subtitle=labels['subtitle'],
)
plt.show()

In [ ]:
# Categoricas e binarias
from src.viz import plot_univariate_categorical

labels = lang({
    'title': 'Distribuições univariadas — categóricas e binárias',
    'subtitle': 'mens e womens não são mutuamente excludentes — ambos ≈55%',
})
fig, axes = plot_univariate_categorical(
    df, BIN_VARS, CAT_VARS, title=labels['title'], subtitle=labels['subtitle'],
    rotate_vars=['history_segment'],
)
plt.show()

In the observed sample, `recency` is approximately uniform between 1 and 12 months, without over-representation of very recent or older buyers. The `history` variable is strongly right-skewed, with most customers spending under \$200 and few exceeding \$1,000, suggesting a log transformation for linear models while tree-based models require none. Both `mens` and `womens` indicators occur at roughly 55%, indicating overlap across categories, whereas `history_segment` discretizes `history` into levels redundant for tree models.

<a id='s2-3'></a>
## 2.3 Randomization audit

This step audits the integrity of randomization. In a properly executed RCT, pre-treatment covariates distribute similarly across arms within the margin expected from random sampling.

Two complementary approaches assess covariate balance: Standardized Mean Difference (SMD) (Austin, 2009), a standardized effect-size metric where $|SMD| < 0.10$ indicates acceptable balance and $|SMD| < 0.05$ indicates high balance, preferred over p-values because it is insensitive to sample size; and formal hypothesis tests (ANOVA F for continuous variables and $\chi^2$ for categorical variables) as a secondary check where $p > 0.05$ is expected across all covariates.

In [ ]:
from src.eda import build_smd_table

smd_mens = build_smd_table(df, TREATMENT, CONTROL_ARM, 'Mens E-Mail',
                            CONT_VARS, BIN_VARS, CAT_VARS)
smd_womens = build_smd_table(df, TREATMENT, CONTROL_ARM, 'Womens E-Mail',
                              CONT_VARS, BIN_VARS, CAT_VARS)

labels = lang({
    'balance_header': 'Balance: Mens E-Mail vs No E-Mail',
    'max_smd': 'Max |SMD|',
    'vars_over': 'Variáveis com |SMD| > 0.05',
    'of': 'de',
})
print(f"=== {labels['balance_header']} ===")
print(smd_mens.round(4).to_string(index=False))
print(f"\n{labels['max_smd']}: {smd_mens['smd'].abs().max():.4f}")
print(f"{labels['vars_over']}: {(smd_mens['smd'].abs() > 0.05).sum()} {labels['of']} {len(smd_mens)}")

In [ ]:
labels = lang({
    'balance_header': 'Balance: Womens E-Mail vs No E-Mail',
    'max_smd': 'Max |SMD|',
    'vars_over': 'Variáveis com |SMD| > 0.05',
    'of': 'de',
})
print(f"=== {labels['balance_header']} ===")
print(smd_womens.round(4).to_string(index=False))
print(f"\n{labels['max_smd']}: {smd_womens['smd'].abs().max():.4f}")
print(f"{labels['vars_over']}: {(smd_womens['smd'].abs() > 0.05).sum()} {labels['of']} {len(smd_womens)}")

In [ ]:
from src.viz import plot_love_plot

labels = lang({
    'title': 'Balance covariáveis pré-tratamento',
    'subtitle': 'Todos os |SMD| abaixo de 0.017 — balance excelente nos dois braços',
})
fig, ax = plot_love_plot(
    {'Mens E-Mail vs Control': smd_mens, 'Womens E-Mail vs Control': smd_womens},
    title=labels['title'],
    xlabel='Standardized Mean Difference',
    subtitle=labels['subtitle'],
)
plt.show()

In [ ]:
from src.eda import formal_balance_tests

formal_tests = formal_balance_tests(df, TREATMENT, ARMS, CONT_VARS, BIN_VARS + CAT_VARS)
labels = lang({
    'header': 'Testes de balance (3-way: No / Mens / Womens)',
    'continuous': 'Contínuas (ANOVA F)',
    'categorical': 'Categóricas e binárias (chi-squared)',
})
print(f"=== {labels['header']} ===\n")
print(f"-- {labels['continuous']} --")
print(formal_tests[formal_tests['test'] == 'ANOVA F'].to_string(index=False))
print(f"\n-- {labels['categorical']} --")
print(formal_tests[formal_tests['test'] == 'chi-squared'].to_string(index=False))

All SMD values remained below 0.05 across both comparisons (Mens vs Control and Womens vs Control), with a maximum around 0.01, an order of magnitude below the reference threshold. No formal test rejected the null hypothesis of equal distribution (all $p > 0.3$), confirming internal validity so the following ATEs can be interpreted causally without adjustment for observable confounding.

In the previous project (Marketing Campaign Optimization), `language_preferred` and `marketing_channel` were imbalanced across arms, which required propensity adjustment; in this RCT, randomization removes the need for that adjustment.

<a id='s2-4'></a>
## 2.4 Outcomes: distribution and raw rates

Before estimating ATEs with confidence intervals, raw outcome rates and means are summarized by arm.

In [ ]:
from src.eda import outcome_summary_by_arm

outcomes_by_arm = outcome_summary_by_arm(df, TREATMENT, ARMS)
print(outcomes_by_arm)

In [ ]:
from src.viz import plot_outcomes_by_arm

labels = lang({
    'visit_rate': 'Visit rate',
    'conversion_rate': 'Conversion rate',
    'spend_mean': 'Spend médio ($)',
    'per_arm_suffix': 'por braço',
    'title': 'Outcomes por braço',
    'subtitle': 'Mens E-Mail lidera nos três outcomes; Womens vem em segundo',
})
fig, axes = plot_outcomes_by_arm(
    df, TREATMENT, ARMS,
    outcomes=['visit', 'conversion', 'spend'],
    ylabels=[labels['visit_rate'], labels['conversion_rate'], labels['spend_mean']],
    titles=[
        f"{labels['visit_rate']} {labels['per_arm_suffix']}",
        f"{labels['conversion_rate']} {labels['per_arm_suffix']}",
        f"{labels['spend_mean']} {labels['per_arm_suffix']}",
    ],
    title=labels['title'], subtitle=labels['subtitle'],
)
plt.show()

### Diagnosis of outcome `conversion` (rare class)

Only 578 conversions occurred across 64,000 observations (~0.9%): roughly 122 conversions among 21,306 customers in No E-Mail (0.57%), 267 among 21,307 in Mens E-Mail (1.25%), and 189 among 21,387 in Womens E-Mail (0.88%). When conditioning on relevant subgroups (for example, customers with `recency <= 3`), event counts per subgroup and arm drop to a few dozen, inflating the variance of CATE estimates on `conversion`. That limitation provides the empirical justification for choosing `visit` (9,394 total events, ~14.7%) as the primary outcome.

<a id='s2-5'></a>
## 2.5 ATE with confidence intervals

ATEs are estimated as differences in means between arms, a procedure justified by the randomization verified in Section 2.3 and insufficient in observational designs. For inference, binary outcomes (`visit` and `conversion`) use normal confidence intervals based on the standard error of the difference in proportions, while the continuous outcome (`spend`) uses Welch's t-test to allow unequal variances.

In [ ]:
from src.effects import ate_table

ate_results = ate_table(
    df, TREATMENT, TREATED_ARMS, CONTROL_ARM,
    binary_outcomes=['visit', 'conversion'], continuous_outcome='spend',
)
print(ate_results.to_string(index=False))

In [ ]:
from src.viz import plot_ate_forest

labels = lang({
    'title': 'Efeito do tratamento (ATE) por outcome',
    'subtitle': 'Efeito positivo e significativo (p < 0.001) em ambos os braços, nos três outcomes',
})
fig, axes = plot_ate_forest(
    ate_results, outcomes=['visit', 'conversion', 'spend'],
    xlabel_prefix='ATE: ', title_prefix='Effect of treatment on ',
    title=labels['title'], subtitle=labels['subtitle'],
)
plt.show()

Across all outcomes and both treated arms, the average effect was positive and statistically significant ($p < 0.001$). For `visit`, Mens E-Mail raised the rate by +7.66pp and Womens E-Mail by +4.52pp, with non-overlapping confidence intervals. For `conversion`, gains were +0.68pp under Mens E-Mail and +0.31pp under Womens E-Mail, also with separated intervals. For `spend`, Mens E-Mail generated ~\$0.77 in incremental revenue per customer and Womens E-Mail ~\$0.42, corresponding to the difference between \$770K and \$420K across a hypothetical 1M-customer base.

Mens E-Mail leading Womens E-Mail on population average still leaves room for individual heterogeneity: subgroups may exist, such as customers who purchase only in the `womens` category, where Womens E-Mail is more effective. That hypothesis motivates uplift modeling to capture heterogeneous effects masked by averages.

<a id='s2-6'></a>
## 2.6 Synthesis of internal validity

This section confirmed three conditions required before CATE estimation:

| Question | Answer | Evidence |
|---|---|---|
| Do the arms have comparable sizes? | Yes, ~33% each | $\chi^2$ against uniform accepts equality |
| Are the covariates balanced? | Yes, at a high level | All \|SMD\| < 0.015; all formal tests with $p > 0.3$ |
| Is there a treatment signal? | Yes, across all outcomes | All ATEs significant at $p < 0.001$ |

In practical terms, unconfoundedness holds by design without requiring propensity adjustment, CATE is identified without bias as $\tau(x) = E[Y \mid T=1, X=x] - E[Y \mid T=0, X=x]$, and the methods in Section 4 (S/T/X/R-learner) and Section 5 (Causal Forest and Uplift Trees) estimate a causal quantity free of observable confounding.

Next: [Section 3: response-targeting baseline model](02_Baseline_Propensity_EN.ipynb) (benchmark for comparison with uplift).

## References

Austin, P. C. (2009). Balance diagnostics for comparing the distribution of baseline covariates between treatment groups in propensity-score matched samples. *Statistics in Medicine*, 28(25), 3083–3107. DOI: [10.1002/sim.3697](https://doi.org/10.1002/sim.3697)

Devriendt, F., Moldovan, D., & Verbeke, W. (2018). A literature survey and experimental evaluation of the state-of-the-art in uplift modeling: A stepping stone toward the development of prescriptive analytics. *Big Data*, 6(1), 13–41. DOI: [10.1089/big.2017.0104](https://doi.org/10.1089/big.2017.0104)

Diemert, E., Betlei, A., Renaudin, C., & Amini, M.-R. (2018). A large scale benchmark for uplift modeling. In *Proceedings of the AdKDD and TargetAd Workshop, KDD 2018*. [https://hal.archives-ouvertes.fr/hal-02515860/](https://hal.archives-ouvertes.fr/hal-02515860/)

Gutierrez, P., & Gérardy, J.-Y. (2017). Causal inference and uplift modelling: A review of the literature. *Proceedings of Machine Learning Research*, 67, 1–13. [https://proceedings.mlr.press/v67/gutierrez17a.html](https://proceedings.mlr.press/v67/gutierrez17a.html)

Hillstrom, K. (2008). *The MineThatData E-Mail Analytics and Data Mining Challenge*. MineThatData blog. [https://blog.minethatdata.com/2008/03/minethatdata-e-mail-analytics-and-data.html](https://blog.minethatdata.com/2008/03/minethatdata-e-mail-analytics-and-data.html)

Holland, P. W. (1986). Statistics and causal inference. *Journal of the American Statistical Association*, 81(396), 945–960. DOI: [10.1080/01621459.1986.10478354](https://doi.org/10.1080/01621459.1986.10478354)

Imbens, G. W., & Rubin, D. B. (2015). *Causal Inference for Statistics, Social, and Biomedical Sciences: An Introduction*. Cambridge University Press. DOI: [10.1017/CBO9781139025751](https://doi.org/10.1017/CBO9781139025751)

Radcliffe, N. J., & Surry, P. D. (2011). *Real-World Uplift Modelling with Significance-Based Uplift Trees* (White Paper TR-2011-1). Stochastic Solutions. [https://stochasticsolutions.com/pdf/sig-based-up-trees.pdf](https://stochasticsolutions.com/pdf/sig-based-up-trees.pdf)

Rubin, D. B. (1974). Estimating causal effects of treatments in randomized and nonrandomized studies. *Journal of Educational Psychology*, 66(5), 688–701. DOI: [10.1037/h0037350](https://doi.org/10.1037/h0037350)
